# Objectives

- Transform a wide dataset into a long format using pandas.
- Validate that no information is lost during transformation.
- Handle missing values.
- Prepare a dataset suitable for SQL analysis and visualization.

In [2]:
import pandas as pd
import numpy as np

In [3]:
excel_file = ("../data/raw/WDIEXCEL.xlsx")

In [4]:
df_data = pd.read_excel(excel_file, sheet_name="Data")

In [5]:
df_long = pd.melt(df_data, id_vars=df_data.columns[:4], value_vars=df_data.columns[4:], var_name="Year", value_name="Value")

In [6]:
df_long.shape

(26200020, 6)

In [7]:
df_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 26200020 entries, 0 to 26200019
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   Country Name    str    
 1   Country Code    str    
 2   Indicator Name  str    
 3   Indicator Code  str    
 4   Year            str    
 5   Value           float64
dtypes: float64(1), str(5)
memory usage: 1.2 GB


In [8]:
df_long.head()

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1960,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1960,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,1960,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,1960,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,1960,NaN


In [47]:
df_long["Indicator Name"].str.len().max()
df_long["Country Name"].str.len().max()

np.int64(73)

In [14]:
df_long.sample(5)
df_long.sample(5, random_state=42)

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
15511791,IDA total,IDA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1999,NaN
24931026,San Marino,SMR,"Self-employed, total (% of total employment) (...",SL.EMP.SELF.ZS,2022,NaN
15765597,New Zealand,NZL,Imports of goods and services (constant 2015 US$),NE.IMP.GNFS.KD,1999,2.308552e+10
16988554,Rwanda,RWA,Ratio of female to male labor force participat...,SL.TLF.CACT.FM.ZS,2002,7.597062e+01
12771600,Sub-Saharan Africa (IDA & IBRD countries),TSS,Present value of external debt (current US$),DT.DOD.PVLX.CD,1992,NaN


In [15]:
df_long.isnull().sum()

Country Name             0
Country Code             0
Indicator Name           0
Indicator Code           0
Year                     0
Value             17184106
dtype: int64

In [16]:
df_long["Value"].isnull().sum()

np.int64(17184106)

In [17]:
df_long["Value"].isna().sum()

np.int64(17184106)

In [18]:
df_long["Value"].notna().sum()

np.int64(9015914)

# Observations:

* The actual dataset had 396,970 rows & 70 columns.
* We kept first 4 columns fixed & transformed the remaining 66. 
* So, 396,970 × 66 = 26,200,020 is the current number of rows.
* Total observations created: 26,200,020 | Useful observations: 9,015,914 (34%) | Empty observations: 17,184,106


In [37]:
useful_data_percentage = (df_long["Value"].notna().mean()*100)
print(f"Useful Data in percentage: {round(useful_data_percentage)} %")  
# Executed after deleting nulls, so it is giving 100%

Useful Data in percentage: 100 %


In [22]:
df_long = df_long.dropna(subset=["Value"])

In [23]:
df_long.shape

(9015914, 6)

In [24]:
df_long.duplicated().sum()

np.int64(0)

In [25]:
df_long.isnull().sum()

Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0
Year              0
Value             0
dtype: int64

In [26]:
df_long.info()

<class 'pandas.DataFrame'>
Index: 9015914 entries, 50 to 26200006
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   Country Name    str    
 1   Country Code    str    
 2   Indicator Name  str    
 3   Indicator Code  str    
 4   Year            str    
 5   Value           float64
dtypes: float64(1), str(5)
memory usage: 481.5 MB


In [27]:
df_long["Year"] = df_long["Year"].astype(int)

In [28]:
df_long.info()

<class 'pandas.DataFrame'>
Index: 9015914 entries, 50 to 26200006
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   Country Name    str    
 1   Country Code    str    
 2   Indicator Name  str    
 3   Indicator Code  str    
 4   Year            int64  
 5   Value           float64
dtypes: float64(1), int64(1), str(4)
memory usage: 481.5 MB


In [38]:
df_long["Year"].min(), df_long["Year"].max()

(np.int64(1960), np.int64(2025))

In [39]:
df_long["Value"].describe()

count    9.015914e+06
mean     4.320475e+15
std      8.953311e+18
min     -4.178515e+17
25%      4.984017e+00
50%      3.954098e+01
75%      4.830330e+04
max      2.435800e+22
Name: Value, dtype: float64

In [41]:
df_long.loc[df_long["Value"]==df_long["Value"].min()]

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value
25622703,Italy,ITA,Net foreign assets (current LCU),FM.AST.NFRG.CN,2024,-4.178515e+17


In [42]:
df_long.to_csv("../data/cleaned/cleaned_dataset_long_format.csv", index=False)

In [44]:
import os
os.path.exists("../data/cleaned/cleaned_dataset_long_format.csv")

True

In [45]:
os.path.exists("../data/cleaned/world_bank_long_format.csv")

True